In [ ]:
import pandas as pd
import json

X = pd.read_parquet("/context/data/X.parquet")
y = pd.read_parquet("/context/data/y.parquet")

with open("/context/data/moons_split.json") as f:
    splits = json.load(f)

X_train = X[X["moon"].isin(splits["train"])]
y_train = y[y["moon"].isin(splits["train"])]

X_test_cloud = X[X["moon"].isin(splits["reduced_cloud"])]
y_test_cloud = y[y["moon"].isin(splits["reduced_cloud"])]


In [ ]:
import joblib
import os
import time
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.linear_model import ElasticNet
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_regression
import lightgbm as lgb
import xgboost as xgb

def train(X_train, y_train, model_directory_path, loop_moon, embargo):
    t_start = time.perf_counter()

    # ── Config ────────────────────────────────────────────────────────────────
    VAL_WINDOW = 15
    MIN_TRAIN_MOONS = 20
    N_FEATURES_SELECT = 500

    LOOKBACKS = [50, 100, 200]
    ENET_PARAMS_LIST = [{"alpha": 0.0005, "l1_ratio": 0.5},
                        {"alpha": 0.001, "l1_ratio": 0.3}]
    LGB_PARAMS_LIST = [{"num_leaves": 200, "learning_rate": 0.02,
                        "feature_fraction": 0.8, "bagging_fraction": 0.8,
                        "reg_alpha": 0.1, "reg_lambda": 0.1},
                       {"num_leaves": 300, "learning_rate": 0.01,
                        "feature_fraction": 1.0, "bagging_fraction": 1.0,
                        "reg_alpha": 0.0, "reg_lambda": 0.0}]

    # ── Utilities ──────────────────────────────────────────────────────────────
    def fast_cs_rank(df, cols):
        df = df.copy()
        out = np.empty((len(df), len(cols)), dtype=np.float32)
        for _, idx in df.groupby("moon", sort=False).groups.items():
            ia = idx.values
            block = df.iloc[ia][cols].values.astype(np.float32)
            n = len(ia)
            if n < 2:
                out[ia] = 0.0
                continue
            ranks = np.argsort(np.argsort(block, axis=0), axis=0).astype(np.float32)
            out[ia] = ranks / (n - 1) - 0.5
        for i, c in enumerate(cols):
            df[c] = out[:, i]
        return df

    def fill_median(df, cols):
        df = df.copy()
        medians = df.groupby("moon")[cols].transform("median")
        for c in cols:
            mask = df[c].isna()
            df.loc[mask, c] = medians.loc[mask, c]
        df[cols] = df[cols].fillna(0.0)
        return df

    def moon_spearman(yt, yp):
        if len(yt) < 5:
            return 0.0
        if np.unique(yt).size < 2 or np.unique(yp).size < 2:
            return 0.0
        r, _ = spearmanr(yp, yt)
        return float(r) if not np.isnan(r) else 0.0

    # ── Prepare data ───────────────────────────────────────────────────────────
    print("  [1/5] Preparing data ...")
    df = X_train.merge(y_train[["moon", "id", "target"]], on=["moon", "id"])
    df = df.dropna(subset=["target"])

    ALL_FEATS = [c for c in df.columns if c.startswith("Feature_") and df[c].nunique() > 1]
    print(f"  Using {len(ALL_FEATS)} raw feature columns")

    df = fill_median(df, ALL_FEATS)
    df = fast_cs_rank(df, ALL_FEATS)          # rank features within each moon
    # Keep raw target (no ranking of target)

    # Feature selection
    X_all = df[ALL_FEATS].values.astype(np.float32)
    y_all = df["target"].values.astype(np.float32)   # raw target

    selector_var = VarianceThreshold(threshold=0.001)
    X_var = selector_var.fit_transform(X_all)
    kept_feats = [ALL_FEATS[i] for i in range(len(ALL_FEATS)) if selector_var.get_support()[i]]
    print(f"  After variance threshold: {len(kept_feats)} features")

    selector_kbest = SelectKBest(f_regression, k=min(N_FEATURES_SELECT, len(kept_feats)))
    X_selected = selector_kbest.fit_transform(X_var, y_all)
    selected_feats = [kept_feats[i] for i in range(len(kept_feats)) if selector_kbest.get_support()[i]]
    print(f"  Final selected features: {len(selected_feats)}")

    # Rebuild dataframe (avoid fragmentation)
    df_selected = pd.concat([df[["moon", "target"]],
                             pd.DataFrame(X_selected, columns=selected_feats, index=df.index)],
                            axis=1)
    ALL_FEATS = selected_feats

    # ── Rolling-origin validation ──────────────────────────────────────────────
    all_moons = sorted(df_selected["moon"].unique())
    print(f"  Moons: {len(all_moons)}  ({all_moons[0]}→{all_moons[-1]})")

    best_score = -np.inf
    best_params = {"lookback": None, "enet": None, "lgb": None}
    best_models = None

    for lb in LOOKBACKS:
        if len(all_moons) - lb - VAL_WINDOW < 0:
            train_moons = all_moons[:-VAL_WINDOW]
        else:
            train_moons = all_moons[-lb-VAL_WINDOW:-VAL_WINDOW] if lb > 0 else all_moons[:-VAL_WINDOW]
        val_moons = all_moons[-VAL_WINDOW:]

        if len(train_moons) < MIN_TRAIN_MOONS:
            continue

        train_mask = df_selected["moon"].isin(train_moons)
        val_mask = df_selected["moon"].isin(val_moons)

        X_tr = df_selected.loc[train_mask, ALL_FEATS].values.astype(np.float32)
        y_tr = df_selected.loc[train_mask, "target"].values
        X_val = df_selected.loc[val_mask, ALL_FEATS].values.astype(np.float32)
        y_val_raw = df_selected.loc[val_mask, "target"].values
        moon_val = df_selected.loc[val_mask, "moon"].values

        for enet_p in ENET_PARAMS_LIST:
            for lgb_p in LGB_PARAMS_LIST:
                enet = ElasticNet(**enet_p, max_iter=2000, random_state=42, fit_intercept=False)
                enet.fit(X_tr, y_tr)
                lgb_model = lgb.LGBMRegressor(**lgb_p, n_estimators=500, random_state=42, verbose=-1)
                lgb_model.fit(X_tr, y_tr)

                pred_enet = enet.predict(X_val)
                pred_lgb = lgb_model.predict(X_val)
                pred_ensemble = 0.5 * pred_enet + 0.5 * pred_lgb

                ics = []
                for mv in val_moons:
                    mask = (moon_val == mv)
                    if mask.sum() >= 5:
                        ics.append(moon_spearman(y_val_raw[mask], pred_ensemble[mask]))
                score = np.mean(ics) if ics else -999.0

                if score > best_score:
                    best_score = score
                    best_params = {"lookback": lb, "enet": enet_p, "lgb": lgb_p}
                    best_models = (enet, lgb_model)
                    print(f"    New best: lb={lb} | enet={enet_p} | lgb={lgb_p} | CV={score:.5f}")

    if best_models is None:
        raise RuntimeError("No valid model found during CV")

    best_enet, best_lgb = best_models
    lb = best_params["lookback"]
    train_moons_final = all_moons[-lb:] if lb < len(all_moons) else all_moons
    final_mask = df_selected["moon"].isin(train_moons_final)
    X_final = df_selected.loc[final_mask, ALL_FEATS].values.astype(np.float32)
    y_final = df_selected.loc[final_mask, "target"].values

    print(f"  [3/5] Final training on {len(train_moons_final)} moons ({len(X_final):,} rows) ...")
    best_enet.fit(X_final, y_final)
    best_lgb.fit(X_final, y_final)

    xgb_model = xgb.XGBRegressor(objective="reg:squarederror", max_depth=5,
                                 learning_rate=0.02, n_estimators=500,
                                 colsample_bytree=0.8, random_state=42)
    xgb_model.fit(X_final, y_final)

    # ── Save everything ─────────────────────────────────────────────────────────
    os.makedirs(model_directory_path, exist_ok=True)
    joblib.dump({
        "enet": best_enet,
        "lgb": best_lgb,
        "xgb": xgb_model,
        "selected_features": ALL_FEATS,
        "original_features": [c for c in X_train.columns if c.startswith("Feature_")],
        "selector_var": selector_var,
        "selector_kbest": selector_kbest,
        "best_cv": best_score,
    }, os.path.join(model_directory_path, "model_ensemble.joblib"))

    print(f"  [5/5] Done in {time.perf_counter()-t_start:.1f}s | saved.")

In [ ]:
def infer(X_test, model_directory_path, loop_moon, embargo):
    obj = joblib.load(os.path.join(model_directory_path, "model_ensemble.joblib"))
    enet = obj["enet"]
    lgb_model = obj["lgb"]
    xgb_model = obj["xgb"]
    selected_feats = obj["selected_features"]
    original_feats = obj["original_features"]
    selector_var = obj["selector_var"]
    selector_kbest = obj["selector_kbest"]

    X = X_test.copy()

    # Ensure all original features exist
    for c in original_feats:
        if c not in X.columns:
            X[c] = 0.0
    X_full = X[original_feats].values.astype(np.float32)

    # Cross‑sectional rank features within each moon (same as training)
    X_ranked = np.empty_like(X_full, dtype=np.float32)
    moon_vals = X["moon"].values
    for moon in np.unique(moon_vals):
        mask = (moon_vals == moon)
        if mask.sum() < 2:
            X_ranked[mask] = 0.0
            continue
        block = X_full[mask]
        ranks = np.argsort(np.argsort(block, axis=0), axis=0).astype(np.float32)
        X_ranked[mask] = ranks / (mask.sum() - 1) - 0.5

    # Apply feature selection
    X_var = selector_var.transform(X_ranked)
    X_selected = selector_kbest.transform(X_var)

    # Raw predictions from each model
    pred_enet = enet.predict(X_selected)
    pred_lgb = lgb_model.predict(X_selected)
    pred_xgb = xgb_model.predict(X_selected)

    # Simple ensemble (weights can be tuned)
    pred_ensemble = 0.4 * pred_enet + 0.3 * pred_lgb + 0.3 * pred_xgb

    # No final ranking – output raw predictions as is
    return pd.DataFrame({
        "moon": X_test["moon"].values,
        "id": X_test["id"].values,
        "prediction": pred_ensemble,
    })

In [ ]:
embargo = 4
model_dir = "./model_cr3"
os.makedirs(model_dir, exist_ok=True)

train(X_train, y_train, model_dir, loop_moon=0, embargo=embargo)

results = []
for moon in splits["reduced_cloud"]:
    X_moon = X_test_cloud[X_test_cloud["moon"] == moon]
    pred = infer(X_moon, model_dir, loop_moon=moon, embargo=embargo)
    results.append(pred)

predictions = pd.concat(results)

from scipy.stats import pearsonr

for moon in splits["reduced_cloud"]:
    p  = predictions[predictions["moon"] == moon]
    gt = y_test_cloud[y_test_cloud["moon"] == moon]
    merged = p.merge(gt, on=["moon", "id"])
    if len(merged) > 1:
        corr, _ = pearsonr(merged["prediction"], merged["target"])
        print(f"Moon {moon}: Pearson r = {corr:.4f}")
